# NCM Sudoku — Aşama 0 ve 1: Yeniden üretim + PTQ taraması

**Amaç.** Resmî NCM Sudoku modelini (25M parametreli üreteç, 12 seviye) yükleyip makaledeki sonucu yeniden üretmek
(zor seviyede ≈%86.1 tamamen çözülen tahta), ardından üretecin ağırlıklarını **yeniden eğitmeden** INT8, INT4 ve
ternary (1.58-bit) değerlere çevirip kalitenin nasıl değiştiğini ölçmek.

**Araştırma sorusunun ilk yarısı:** *"Kabadan inceye üretimde eğitim sonrası ternary kuantalama çöker mi?"*

**Başlamadan önce:** Menüden `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seç.

Notlar:
- Resmî repo dosyalarına dokunulmaz; ağırlıklar bellekte kuantalanır.
- Yalnızca üreteç kuantalanır, tokenizer (VQ-VAE) tam hassasiyette kalır.
- Sonuçlar Google Drive'a yazılır; oturum koparsa aynı hücreyi tekrar çalıştırınca kaldığı yerden devam eder.

## 1. GPU ve Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/ncm_ptq'
os.makedirs(OUT_DIR, exist_ok=True)
print('Sonuçlar:', OUT_DIR)

## 2. Resmî repo, bağımlılıklar ve Sudoku checkpoint'i (yalnızca `sudoku/hard`, ≈160 MB)

In [ ]:
%cd /content
!test -d ncm || git clone https://github.com/HazyResearch/ncm.git
%cd /content/ncm
!git rev-parse HEAD > {OUT_DIR}/ncm_commit.txt && cat {OUT_DIR}/ncm_commit.txt
!pip install -q -U omegaconf hydra-core "transformers>=4.51" "huggingface_hub[cli]" regex rouge_score sacrebleu
!hf download hazyresearch/ncm-checkpoints --include "sudoku/hard/*" --local-dir artifacts
!hf download hazyresearch/ncm-datasets --repo-type dataset --include "sudoku/hard/*" --local-dir data
!ls -la artifacts/sudoku/hard data/sudoku/hard

## 3. Aşama 0a — Resmî değerlendirme betiğiyle yeniden üretim

Hedef: `solved %` ≈ 86.1 (makale Tablo 3, zor seviye). Checkpoint'ler `artifacts/`, bulmacalar ve çözümler `data/`
klasöründe.

In [ ]:
%cd /content/ncm
!python evaluate.py --config config/eval/sudoku-hard.yaml --checkpoint-dir artifacts/sudoku/hard --data-dir data/sudoku/hard 2>&1 | tail -n 8

## 4. PTQ modülünü yaz (`ncm_ptq.py`)

In [ ]:
%%writefile /content/ncm_ptq.py
"""
NCM Sudoku — Aşama 0/1: resmî checkpoint'i yeniden üret ve eğitim sonrası kuantalama (PTQ) taraması.

Resmî HazyResearch/ncm reposundaki scripts/eval_shard.py (Sudoku dalı) ve
scripts/eval_merge.py::exact() mantığını tek süreçte, değiştirmeden taklit eder.
Repo dosyalarına dokunmaz; ağırlıklar yüklendikten sonra bellekte kuantalanır.

Yalnızca üreteç (NextConceptTransformer) kuantalanır. Tokenizer (VQ-VAE) tam hassasiyette kalır,
çünkü üreteç onu `vqvae_local` adlı bir tuple içinde tutar ve net.modules() onu kapsamaz.
"""

from __future__ import annotations

import copy
import json
import os
import pickle
import sys
import time
from dataclasses import dataclass, asdict
from typing import Callable, Dict, List, Optional

import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1. Ağırlık kuantalayıcıları (fake-quant: değerler kuantalanır, dtype float kalır)
#    Ağırlık şekli nn.Linear için [out_features, in_features]; "kanal" = çıkış satırı.
# ---------------------------------------------------------------------------

def q_int_per_channel(w: torch.Tensor, bits: int) -> torch.Tensor:
    """Simetrik absmax, satır başına ölçek. bits=8 -> [-127,127], bits=4 -> [-7,7]."""
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
    return (w / scale).round().clamp(-qmax, qmax) * scale


def q_ternary_absmean_tensor(w: torch.Tensor) -> torch.Tensor:
    """BitNet b1.58: gamma = mean|W| (tüm tensör), T = RoundClip(W/gamma, -1, 1), W' = gamma*T."""
    gamma = w.abs().mean().clamp(min=1e-8)
    return (w / gamma).round().clamp(-1, 1) * gamma


def q_ternary_absmean_channel(w: torch.Tensor) -> torch.Tensor:
    """Absmean ternary, satır başına gamma."""
    gamma = w.abs().mean(dim=1, keepdim=True).clamp(min=1e-8)
    return (w / gamma).round().clamp(-1, 1) * gamma


def q_ternary_twn_channel(w: torch.Tensor) -> torch.Tensor:
    """
    Ternary Weight Networks (Li & Liu, 2016) eşiği, satır başına:
        delta = 0.7 * mean|w|,  T = sign(w) * [|w| > delta]
        alpha = mean_{|w|>delta} |w|   (destek üzerinde L2-optimal ölçek)
    Absmean'den farkı: ölçek sıfır olmayan ağırlıklara göre yeniden hesaplanır.
    """
    a = w.abs()
    delta = 0.7 * a.mean(dim=1, keepdim=True)
    mask = (a > delta).to(w.dtype)
    alpha = (a * mask).sum(dim=1, keepdim=True) / mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    return torch.sign(w) * mask * alpha


WEIGHT_QUANTIZERS: Dict[str, Callable[[torch.Tensor], torch.Tensor]] = {
    "int8_ch": lambda w: q_int_per_channel(w, 8),
    "int4_ch": lambda w: q_int_per_channel(w, 4),
    "tern_absmean_tensor": q_ternary_absmean_tensor,
    "tern_absmean_ch": q_ternary_absmean_channel,
    "tern_twn_ch": q_ternary_twn_channel,
}


def act_quant_int8_per_token(x: torch.Tensor) -> torch.Tensor:
    """BitNet tarzı aktivasyon kuantalaması: token başına absmax, [-127,127]."""
    scale = x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-5) / 127.0
    return (x / scale).round().clamp(-127, 127) * scale


# ---------------------------------------------------------------------------
# 2. Kapsam (scope): hangi lineer katmanlar kuantalanacak
#    Üreteçteki adlar: transformer.wte (giriş izdüşümü), transformer.blocks.{i}.attn.c_attn / c_proj,
#    transformer.blocks.{i}.mlp.c_fc / c_proj, lm_heads.{k} (seviye başına çıkış katmanı).
#    Gerçek adlar notebook'ta list_linear_layers() ile yazdırılır; buradaki filtreler alt-dize eşleşmesidir.
# ---------------------------------------------------------------------------

def _is_io(name: str) -> bool:
    return ("wte" in name) or ("lm_head" in name) or ("ctx_" in name)


SCOPES: Dict[str, Callable[[str], bool]] = {
    "all": lambda n: True,
    "body": lambda n: not _is_io(n),             # giriş/çıkış katmanları hariç
    "attn": lambda n: (".attn." in n) and not _is_io(n),
    "mlp": lambda n: (".mlp." in n) and not _is_io(n),
    "io_only": _is_io,                            # yalnızca giriş/çıkış katmanları
}


def list_linear_layers(net: nn.Module) -> List[Dict[str, object]]:
    rows = []
    for name, m in net.named_modules():
        if isinstance(m, nn.Linear):
            rows.append({"name": name, "shape": tuple(m.weight.shape), "params": m.weight.numel(),
                         "bias": m.bias is not None})
    return rows


@dataclass
class QuantReport:
    n_layers: int
    n_params: int
    rel_err: float          # ||W'-W||_F / ||W||_F, kuantalanan katmanlar üzerinden toplam
    zero_frac: float        # kuantalanan ağırlıklarda sıfır oranı (ternary için seyreklik)


def quantize_net(net: nn.Module, weight_scheme: Optional[str], scope: str,
                 act_int8: bool = False) -> QuantReport:
    """Ağırlıkları yerinde kuantalar; act_int8=True ise girişlere forward_pre_hook ile INT8 uygular."""
    sel = SCOPES[scope]
    fn = WEIGHT_QUANTIZERS[weight_scheme] if weight_scheme else None
    n_layers = n_params = 0
    err2 = ref2 = 0.0
    zeros = 0
    for name, m in net.named_modules():
        if not isinstance(m, nn.Linear) or not sel(name):
            continue
        n_layers += 1
        n_params += m.weight.numel()
        if fn is not None:
            with torch.no_grad():
                w = m.weight.data.float()
                wq = fn(w)
                err2 += float((wq - w).pow(2).sum())
                ref2 += float(w.pow(2).sum())
                zeros += int((wq == 0).sum())
                m.weight.data.copy_(wq.to(m.weight.dtype))
        if act_int8:
            m.register_forward_pre_hook(lambda _mod, inp: (act_quant_int8_per_token(inp[0]),) + tuple(inp[1:]))
    return QuantReport(
        n_layers=n_layers,
        n_params=n_params,
        rel_err=(err2 / ref2) ** 0.5 if ref2 > 0 else 0.0,
        zero_frac=zeros / n_params if (fn is not None and n_params) else 0.0,
    )


# ---------------------------------------------------------------------------
# 3. Resmî kodla yükleme ve değerlendirme (eval_shard.py Sudoku dalının birebir karşılığı)
# ---------------------------------------------------------------------------

class SudokuEvaluator:
    def __init__(self, ncm_root: str, config_path: str, ckpt_dir: str, data_dir: str,
                 n: int = 0, batch_size: int = 0, device: Optional[str] = None):
        sys.path.insert(0, ncm_root)
        from omegaconf import OmegaConf
        from models.next_concept import NextConceptTransformer
        from tokenization.vqvae_tokenizer import VQVAETokenizer

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        cfg = OmegaConf.load(config_path)
        assert str(cfg.eval.metric) == "exact", "Bu betik yalnızca Sudoku (metric: exact) içindir"
        self.cfg = cfg
        G = cfg.generation
        self.temp = float(G.get("temperature", 1.0))
        self.top_k = int(G.get("top_k", 1))
        self.top_p = float(G.get("top_p", 0.0))
        self.cfg_scale = float(G.get("cfg_scale", 0.0))
        assert not G.get("frequency_penalty") and not G.get("adjacency_penalty"), \
            "Bu config ek örnekleme cezaları kullanıyor; betik bunları desteklemiyor"

        cfg.tokenizer.vqvae.checkpoint_path = os.path.join(ckpt_dir, cfg.checkpoints.tokenizer)
        self.tk = VQVAETokenizer.from_config(cfg.tokenizer)
        self.tk.vqvae = self.tk.vqvae.to(self.device).eval()
        self.net = NextConceptTransformer(vqvae_local=self.tk.vqvae, config=cfg.model).to(self.device)
        sd = torch.load(os.path.join(ckpt_dir, cfg.checkpoints.generator), map_location="cpu", weights_only=False)
        self.net.load_state_dict(sd["model"], strict=True)
        self.net.eval()
        self.pristine = copy.deepcopy(self.net.state_dict())  # her deney öncesi geri yüklemek için

        self.P = int(cfg.model.prefix_len)
        self.EOS = self.tk.pre_tokenizer.eos_token_id
        self.PAD = self.tk.pre_tokenizer.pad_token_id
        split = cfg.eval.get("split", "test")
        n = n or int(cfg.eval.get("n_docs", 0))
        self.bs = batch_size or int(cfg.eval.batch_size)
        self.items = pickle.load(open(os.path.join(data_dir, cfg.data[split]), "rb"))[:n]
        refs = pickle.load(open(os.path.join(data_dir, cfg.data[f"{split}_refs"]), "rb"))[:n]
        assert len(self.items) == len(refs)
        self.refs = [r if isinstance(r, str) else self._dec(r) for r in refs]

    # --- eval_shard.py ile aynı yardımcılar ---
    def _prefix(self, item):
        ids = list(item)
        return (ids[:self.P] + [self.PAD] * self.P)[:self.P]

    def _dec(self, ids):
        ids = list(ids)
        if self.EOS in ids:
            ids = ids[:ids.index(self.EOS)]
        return self.tk.pre_tokenizer.decode([t for t in ids if t not in (self.EOS, self.PAD)],
                                            skip_special_tokens=True)

    def reset(self):
        """Ağırlıkları orijinaline döndürür ve tüm forward hook'larını temizler."""
        self.net.load_state_dict(self.pristine, strict=True)
        for m in self.net.modules():
            m._forward_pre_hooks.clear()
        self.net.eval()

    @torch.no_grad()
    def generate_all(self) -> List[str]:
        hyps: List[str] = []
        for s in range(0, len(self.items), self.bs):
            x = torch.tensor([self._prefix(it) for it in self.items[s:s + self.bs]], device=self.device)
            g = self.net.generate(None, x, temperature=self.temp, top_k=self.top_k, top_p=self.top_p,
                                  cfg_scale=self.cfg_scale, cfg_null_eos=self.EOS)
            _, zl = self.tk.vqvae.vq.idx_to_zhats(g, return_zlevels=True)
            ids = self.tk.vqvae.decoder(zl[-1]).argmax(-1).tolist()
            hyps += [self._dec(r) for r in ids]
        return hyps

    # --- eval_merge.py::exact() ile aynı skorlama ---
    @staticmethod
    def score(hyps: List[str], refs: List[str]) -> Dict[str, float]:
        def cells(t):
            return t.replace("|", "")

        def valid(t):
            g = cells(t)
            if len(g) != 81 or not g.isdigit() or "0" in g:
                return False
            rows_ = [g[i * 9:(i + 1) * 9] for i in range(9)]
            cols_ = ["".join(r[j] for r in rows_) for j in range(9)]
            boxes = ["".join(rows_[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3))
                     for br in (0, 3, 6) for bc in (0, 3, 6)]
            return all(sorted(x) == list("123456789") for x in rows_ + cols_ + boxes)

        n = len(hyps)
        exact_n = sum(h == r for h, r in zip(hyps, refs))
        cell = sum(sum(a == b for a, b in zip(cells(h), cells(r))) / max(1, len(cells(r)))
                   for h, r in zip(hyps, refs)) / n
        valid_n = sum(valid(h) for h in hyps)
        return {"solved_pct": 100.0 * exact_n / n, "cell_pct": 100.0 * cell,
                "valid_pct": 100.0 * valid_n / n, "n": n}

    def run(self, name: str, weight_scheme: Optional[str], scope: str, act_int8: bool = False) -> Dict:
        self.reset()
        rep = quantize_net(self.net, weight_scheme, scope, act_int8=act_int8)
        t0 = time.perf_counter()
        hyps = self.generate_all()
        if self.device == "cuda":
            torch.cuda.synchronize()
        sec = time.perf_counter() - t0
        out = {"name": name, "weight_scheme": weight_scheme or "fp", "scope": scope,
               "act_int8": act_int8, **asdict(rep), **self.score(hyps, self.refs), "gen_seconds": sec}
        self.reset()
        return out


# ---------------------------------------------------------------------------
# 4. Deney listesi (Aşama 1)
# ---------------------------------------------------------------------------

EXPERIMENTS = [
    # (ad,                         ağırlık şeması,          kapsam,  INT8 aktivasyon)
    ("fp",                          None,                    "all",   False),
    ("fp+a8",                       None,                    "all",   True),
    ("int8_ch/all",                 "int8_ch",               "all",   False),
    ("int4_ch/all",                 "int4_ch",               "all",   False),
    ("int4_ch/body",                "int4_ch",               "body",  False),
    ("tern_absmean_tensor/all",     "tern_absmean_tensor",   "all",   False),
    ("tern_absmean_tensor/body",    "tern_absmean_tensor",   "body",  False),
    ("tern_absmean_tensor/body+a8", "tern_absmean_tensor",   "body",  True),
    ("tern_absmean_ch/body",        "tern_absmean_ch",       "body",  False),
    ("tern_twn_ch/body",            "tern_twn_ch",           "body",  False),
    ("tern_absmean_tensor/attn",    "tern_absmean_tensor",   "attn",  False),
    ("tern_absmean_tensor/mlp",     "tern_absmean_tensor",   "mlp",   False),
    ("tern_absmean_tensor/io_only", "tern_absmean_tensor",   "io_only", False),
]


def run_all(ev: SudokuEvaluator, out_json: str, experiments=EXPERIMENTS) -> List[Dict]:
    """Sonuçları her deneyden sonra diske yazar; dosyada olan deneyleri atlar (oturum koparsa devam)."""
    done: Dict[str, Dict] = {}
    if os.path.exists(out_json):
        for r in json.load(open(out_json)):
            done[r["name"]] = r
    for name, scheme, scope, a8 in experiments:
        if name in done:
            print(f"[atla] {name} (zaten var)")
            continue
        r = ev.run(name, scheme, scope, act_int8=a8)
        done[name] = r
        json.dump(list(done.values()), open(out_json, "w"), indent=2)
        print(f"[{name:30s}] solved={r['solved_pct']:6.2f}%  cell={r['cell_pct']:6.2f}%  "
              f"valid={r['valid_pct']:6.2f}%  rel_err={r['rel_err']:.3f}  zero={r['zero_frac']:.3f}  "
              f"({r['gen_seconds']:.1f}s)")
    return list(done.values())

## 5. Aşama 0b — Kendi değerlendiricimiz resmî sonuçla aynı mı?

Aşağıdaki `fp` satırı, 3. adımdaki resmî çıktıyla **aynı** `solved %` değerini vermeli. Vermezse PTQ sonuçlarına
güvenme; önce farkın nedenini bul.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import ncm_ptq; importlib.reload(ncm_ptq)

ev = ncm_ptq.SudokuEvaluator(
    ncm_root='/content/ncm',
    config_path='/content/ncm/config/eval/sudoku-hard.yaml',
    ckpt_dir='/content/ncm/artifacts/sudoku/hard',
    data_dir='/content/ncm/data/sudoku/hard',
    n=0,            # 0 = config'teki tüm bulmacalar (2000). Hızlı deneme için 256 yaz.
)
print(len(ev.items), 'bulmaca, cihaz =', ev.device)
fp = ev.run('fp-check', None, 'all')
print(fp)

## 6. Üreteçteki lineer katmanlar (kapsam filtrelerini kontrol etmek için)

In [ ]:
import pandas as pd
layers = pd.DataFrame(ncm_ptq.list_linear_layers(ev.net))
for sc, f in ncm_ptq.SCOPES.items():
    layers[sc] = layers['name'].map(f)
print('Toplam lineer parametre:', layers.params.sum())
print(layers.groupby(['body', 'attn', 'mlp', 'io_only']).params.agg(['count', 'sum']))
layers.head(12)

## 7. Aşama 1 — PTQ taraması

| Şema | Açıklama |
|---|---|
| `int8_ch`, `int4_ch` | Simetrik absmax, çıkış kanalı başına ölçek |
| `tern_absmean_tensor` | BitNet b1.58: γ = mean\|W\| (tüm tensör), T = RoundClip(W/γ, −1, 1) |
| `tern_absmean_ch` | Aynı yöntem, kanal başına γ |
| `tern_twn_ch` | TWN eşiği 0.7·mean\|W\| + destek üzerinde yeniden hesaplanan ölçek (en güçlü basit PTQ) |
| `+a8` | Lineer girişlerine token başına INT8 aktivasyon kuantalaması |

| Kapsam | Hangi katmanlar |
|---|---|
| `all` | Tüm lineer katmanlar |
| `body` | Transformer blokları (giriş `wte` ve çıkış `lm_heads` hariç) |
| `attn` / `mlp` | Yalnızca dikkat / yalnızca MLP katmanları |
| `io_only` | Yalnızca giriş ve çıkış katmanları |

Her deney 2000 bulmacada birkaç dakika sürer. Sonuçlar her deneyden sonra Drive'a yazılır.

In [ ]:
OUT_JSON = f'{OUT_DIR}/sudoku_hard_ptq.json'
results = ncm_ptq.run_all(ev, OUT_JSON)

## 8. Sonuç tablosu ve grafik

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame(json.load(open(OUT_JSON)))
cols = ['name', 'solved_pct', 'cell_pct', 'valid_pct', 'rel_err', 'zero_frac', 'n_params', 'gen_seconds']
df = df[cols].round(3)
display(df)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(df['name'], df['solved_pct'], color='#4C72B0')
ax.axvline(float(df.loc[df.name == 'fp', 'solved_pct'].iloc[0]), color='k', ls='--', lw=1, label='FP')
ax.set_xlabel('Tamamen çözülen tahta (%) — Sudoku hard')
ax.invert_yaxis(); ax.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/sudoku_hard_ptq.png', dpi=150)
plt.show()

## 9. Nasıl yorumlanır

- **`fp` ≈ `fp-check` ≈ resmî sonuç** olmalı. Değilse sonraki satırlar anlamsızdır.
- **`int8_ch` ≈ `fp`** beklenir. Belirgin bir düşüş, betikte bir hata olduğuna işaret eder.
- **Ternary satırları** araştırma sorusunun cevabıdır: ciddi düşüş, PTQ'nun yetmediğini ve QAT'nin (Aşama 2)
  gerektiğini gösterir. Düşüş küçükse, bu da yayımlanabilir ve ilginç bir bulgudur.
- **`attn` / `mlp` / `io_only`** satırları hangi katman grubunun en hassas olduğunu gösterir.
- `rel_err` = ‖W′ − W‖ / ‖W‖ ve `zero_frac` = ternary ağırlıklardaki sıfır oranı.
- Bu tek bir checkpoint ve tek bir veri bölümüdür; ilk sonuçlar ilginçse aynı taramayı `sudoku/easy` ve
  `sudoku/medium` için de çalıştır (2. adımdaki iki `--include` desenini ve yolları değiştirmen yeterli).

**Sonraki adım (Aşama 2):** Ternary PTQ çökerse, aynı üreteci BitLinear katmanlarıyla QAT ince ayarı yapmak.
Repoda eğitim kodu olmadığı için kayıp fonksiyonu ve eğitim döngüsü makaledeki Algoritma 1'e göre ayrıca yazılacak.